# DenguePharma Analytics — marts versionados
Selecione o Python 3.12 de analytics/.venv. O notebook consulta os mesmos Parquet/DuckDB da aplicação, sem alterar o legado.

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd
from denguepharma_analytics.storage import default_root, verify_run, run_path
root = default_root()
publications = sorted((root / 'publications').glob('*.json'), key=lambda p: p.stat().st_mtime_ns, reverse=True)
assert publications, 'Prepare e publique uma execução antes de abrir o notebook.'
run_id = publications[0].stem
manifest = verify_run(root, run_id)
manifest['input_hash'], manifest['tables']['mart_panorama']['rows']

## Corpus coletado e estudos revisados
São unidades distintas. Ausência de estudos revisados deve permanecer visível; não promover metadados a evidência aprovada.

In [ ]:
with duckdb.connect(str(run_path(root, run_id) / 'analytics.duckdb'), read_only=True) as db:
    display(db.execute('SELECT * FROM corpus_temporal ORDER BY field, category').fetchdf())
    display(db.execute('SELECT * FROM mart_panorama ORDER BY study_id').fetchdf())
    display(db.execute('SELECT * FROM corpus_completeness ORDER BY source, field').fetchdf())

O eixo A é parcial. Sobrevivência, carga de doença e avaliação da IA ainda estão planejadas. Frequência não mede eficácia, e completude não mede risco de viés.